In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-10 15:44:10.264285: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-10 15:44:10.324373: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-10 15:44:11.453330: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.26.16) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "iterations_threshold": 2,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()

In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-10 15:44:27,369 [DEBUG] [Rain] Rain is initialized
2023-07-10 15:44:27,381 [DEBUG] [Provisioner] Creating coordinator
2023-07-10 15:44:27,392 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-10 15:44:27,398 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-10 15:44:27,403 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-10 15:44:27,420 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-10 15:44:27,439 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-10 15:44:27,462 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-10 15:44:32,926 [INFO] [Provisioner] provisioner is serving
2023-07-10 15:44:32,928 [DEBUG] [Provisioner] Starting coordinator
2023-07-10 15:44:32,931 [INFO] [Coordinator] coordinator is serving
2023-07-10 15:44:32,933 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-10 15:44:32,941 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-10 15:44:32,944 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-10 15:44:32,948 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-10 15:44:32,950 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-10 15:44:32,952 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-10 15:44:32,960 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-10 15:44:32,972 [INFO] [Worker_50151] Worker is running 

Epoch 1/20
Epoch 1/20
Epoch 1/20
157/157 [==============================] - 5s 11ms/step - loss: 0.7087 - accuracy: 0.7775
Epoch 2/20
Epoch 2/20
157/157 [==============================] - 5s 11ms/step - loss: 0.7028 - accuracy: 0.7807
Epoch 2/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2965 - accuracy: 0.9103
Epoch 3/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2281 - accuracy: 0.9311
Epoch 4/20
157/157 [==============================] - 2s 12ms/step - loss: 0.2301 - accuracy: 0.9312
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1909 - accuracy: 0.9414
Epoch 5/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1911 - accuracy: 0.9436
Epoch 5/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1849 - accuracy: 0.9435
Epoch 5/20
157/157 [==============================] - 2s 10ms/step - loss: 0.1675 - accuracy: 0.9500
Epoch 6/20
Epoch 6/20
157/157 [===========================

2023-07-10 15:45:15,297 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-10 15:45:15,298 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-10 15:45:15,302 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-10 15:45:15,306 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider
sending data to divider


2023-07-10 15:45:15,489 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-10 15:45:15,496 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully
2023-07-10 15:45:21,821 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-10 15:45:21,824 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
2023-07-10 15:45:21,941 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully


...

DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-10 15:46:01,553 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-10 15:46:01,561 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_2_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_2_trained.pkl from worker3 successfully
2023-07-10 15:46:01,665 [DEBUG] [DeepLearning] Iteration 2/3 complete.
DEBUG:DeepLearning:Iteration 2/3 complete.
2023-07-10 15:46:01,669 [DEBUG] [DeepLearning] Starting iteration 3/3
DEBUG:DeepLearning:Starting iteration 3/3
2023-07-10 15:46:01,746 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-10 15:46:01,750 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
DEBUG:DividerAmbassador:127.0.0.1:50151
2023-07-10 15:46:01

Epoch 1/20
Epoch 1/20
Epoch 1/20
157/157 [==============================] - 5s 12ms/step - loss: 0.0745 - accuracy: 0.9779
Epoch 2/20
157/157 [==============================] - 5s 12ms/step - loss: 0.0736 - accuracy: 0.9786
Epoch 2/20
157/157 [==============================] - 5s 12ms/step - loss: 0.0710 - accuracy: 0.9783
Epoch 2/20
157/157 [==============================] - 2s 12ms/step - loss: 0.0661 - accuracy: 0.9796
Epoch 3/20
157/157 [==============================] - 2s 12ms/step - loss: 0.0620 - accuracy: 0.9803
Epoch 3/20
157/157 [==============================] - 2s 12ms/step - loss: 0.0508 - accuracy: 0.9826
Epoch 4/20
157/157 [==============================] - 2s 12ms/step - loss: 0.0627 - accuracy: 0.9801
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.0564 - accuracy: 0.9819
Epoch 5/20
157/157 [==============================] - 2s 10ms/step - loss: 0.0490 - accuracy: 0.9837
Epoch 5/20
157/157 [==============================] - 2s 11ms/step - 

2023-07-10 15:46:42,285 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-10 15:46:42,289 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-10 15:46:42,292 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-10 15:46:42,302 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-10 15:46:42,305 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3


sending data to divider
sending data to divider
sending data to divider


2023-07-10 15:46:42,313 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-10 15:46:42,467 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-10 15:46:42,472 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-10 15:46:42,483 [DEBUG] [DividerAmbassador] Downloaded ../../../

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 5ms/step - loss: 0.0733 - accuracy: 0.9828

Test accuracy: 98.3%


In [10]:
del rain

2023-07-10 15:47:03,010 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Worker_50151:Worker stopped serving on port: 50151
2023-07-10 15:47:03,018 [INFO] [Worker_50152] Worker stopped serving on port: 50152
INFO:Worker_50152:Worker stopped serving on port: 50152
2023-07-10 15:47:03,024 [INFO] [Worker_50153] Worker stopped serving on port: 50153
INFO:Worker_50153:Worker stopped serving on port: 50153
2023-07-10 15:47:03,028 [DEBUG] [LocalProvisioner] Workers are deleted
DEBUG:LocalProvisioner:Workers are deleted
2023-07-10 15:47:03,031 [INFO] [Provisioner] provisioner stopped serving
INFO:Provisioner:provisioner stopped serving


In [11]:
model = create_model()
rain = Rain(config, model)

2023-07-10 15:47:06,054 [DEBUG] [Rain] Rain is initialized
2023-07-10 15:47:06,054 [DEBUG] [Rain] Rain is initialized
DEBUG:Rain:Rain is initialized
2023-07-10 15:47:06,065 [DEBUG] [Provisioner] Creating coordinator
2023-07-10 15:47:06,065 [DEBUG] [Provisioner] Creating coordinator
DEBUG:Provisioner:Creating coordinator
2023-07-10 15:47:06,079 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
DEBUG:TemporaryFilesManager:Created temporary directory ../../..//RainData/coord/
2023-07-10 15:47:06,087 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-10 15:47:06,087 [DEBUG] [Coordinator] Coordinator is initialized
DEBUG:Coordinator:Coordinator is initialized
2023-07-10 15:47:06,094 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-10 15:47:06,094 [DEBUG] [LocalProvisioner] Provisioner is initialized
DEBUG:LocalProvisioner:Provisioner is initialized
2023-07-10 15:47:06,109 [DEBUG] [TemporaryFilesManager] Created temporary directory ../..

In [12]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-10 15:47:09,552 [INFO] [Provisioner] provisioner is serving
2023-07-10 15:47:09,552 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-10 15:47:09,557 [DEBUG] [Provisioner] Starting coordinator
2023-07-10 15:47:09,557 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-10 15:47:09,563 [INFO] [Coordinator] coordinator is serving
2023-07-10 15:47:09,563 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-10 15:47:09,603 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-10 15:47:09,603 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-10 15:47:09,609 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-10 15:47:09,609 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to de

Epoch 1/20
Epoch 1/20
Epoch 1/20


2023-07-10 15:47:15,825 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-10 15:47:15,825 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Worker_50151:Worker stopped serving on port: 50151
2023-07-10 15:47:15,844 [INFO] [Worker_50152] Worker stopped serving on port: 50152
2023-07-10 15:47:15,844 [INFO] [Worker_50152] Worker stopped serving on port: 50152
INFO:Worker_50152:Worker stopped serving on port: 50152
2023-07-10 15:47:15,863 [INFO] [Worker_50153] Worker stopped serving on port: 50153
2023-07-10 15:47:15,863 [INFO] [Worker_50153] Worker stopped serving on port: 50153
INFO:Worker_50153:Worker stopped serving on port: 50153
2023-07-10 15:47:15,906 [DEBUG] [LocalProvisioner] Workers are deleted
2023-07-10 15:47:15,906 [DEBUG] [LocalProvisioner] Workers are deleted
DEBUG:LocalProvisioner:Workers are deleted
2023-07-10 15:47:15,923 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-10 15:47:15,923 [DEBUG] [DividerAmbassador] divi

157/157 [==============================] - 5s 11ms/step - loss: 0.7115 - accuracy: 0.7761
Epoch 2/20
157/157 [==============================] - 2s 11ms/step - loss: 0.3020 - accuracy: 0.9076
Epoch 3/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2300 - accuracy: 0.9324
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2359 - accuracy: 0.9295
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.2319 - accuracy: 0.9278
Epoch 4/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1852 - accuracy: 0.9447
Epoch 5/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1882 - accuracy: 0.9423
Epoch 5/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1903 - accuracy: 0.9410
Epoch 5/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1662 - accuracy: 0.9510
Epoch 6/20
157/157 [==============================] - 2s 11ms/step - loss: 0.1657 - accuracy: 0.9499
E

2023-07-10 15:47:52,320 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3


...

2023-07-10 15:49:09,637 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-10 15:49:09,658 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-10 15:49:09,658 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


157/157 [==============================] - 2s 11ms/step - loss: 0.0331 - accuracy: 0.9881
sending data to divider


2023-07-10 15:49:09,771 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-10 15:49:09,771 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-10 15:49:09,789 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-10 15:49:09,789 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-10 15:49:09,799 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1


sending data to divider


DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-10 15:49:09,811 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-10 15:49:09,799 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-10 15:49:09,822 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-10 15:49:09,811 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-10 15:49:09,822 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:divider begins downloadin

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [ ]:
del rain

In [ ]:
model = create_model()
rain = Rain(config, model)

In [ ]:
model = rain.train(X_train, y_train, strategy='semi_async')

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [ ]:
del rain